# Train YOLOv8n-cls kiểm tra bao bì — Smart Logistics Sorter (AI-03 / AI-04 / AI-05 / AI-07)

Notebook tương đương `ai/train_yolo_cls.py`, chạy được **từ đầu đến cuối trên Google Colab** (Runtime → GPU T4).

Luồng: mount Drive → cài ultralytics → chuẩn bị dataset theo hộp (`prepare_dataset.py`) → train YOLOv8n-cls 224 / 50 epoch
→ export ONNX opset 12 → `labels.json` + `model_card.json` → đánh giá + chọn ngưỡng an toàn (`evaluate.py`) → chép kết quả về Drive.

Dataset chụp bằng `pi/tools/capture_dataset.py` (AI-01), cấu trúc `dataset/<lớp>/<box_id>_<timestamp>.jpg`, nén thành `dataset.zip`
và đặt ở `MyDrive/smart_sorter/dataset.zip` (hoặc để nguyên thư mục `MyDrive/smart_sorter/dataset`).

## 0. Mount Google Drive và cấu hình

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

CFG = {
    "drive_dir": "/content/drive/MyDrive/smart_sorter",     # chứa dataset.zip hoặc dataset/ ; kết quả cũng chép về đây
    "work_dir": "/content/work" if IN_COLAB else str(Path.cwd() / "work"),
    "repo_url": "https://github.com/<nhom>/smart-sorter.git",  # repo chứa ai/*.py và pi/sorter (sửa cho đúng)
    "repo_dir": "/content/smart-sorter" if IN_COLAB else str(Path.cwd().resolve().parents[0]),
    "classes": 2,            # 2 = ok/defect ; 4 = ok/tear/dent/wet
    "imgsz": 224, "epochs": 50, "patience": 10, "batch": 32, "seed": 42,
    "model": "yolov8n-cls.pt",
    "version": "1.0.0",      # tăng mỗi lần train lại (hiện trên dashboard: model_version)
    "target_recall": 0.95, "min_confidence": 0.6,
}
WORK = Path(CFG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
print("Colab:", IN_COLAB, "| work:", WORK)

## 1. Cài thư viện và lấy mã nguồn `ai/` + `pi/sorter/` (để dùng chung prepare_dataset / evaluate / OnnxClassifier)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.1", "onnx", "onnxruntime", "onnxslim",
                "scikit-learn", "matplotlib", "PyYAML", "opencv-python-headless", "Pillow"], check=True)

REPO = Path(CFG["repo_dir"])
if not (REPO / "ai" / "prepare_dataset.py").exists():
    # Cách 1: clone repo. Cách 2: chép thư mục smart-sorter lên Drive rồi đổi repo_dir ở trên.
    subprocess.run(["git", "clone", "--depth", "1", CFG["repo_url"], str(REPO)], check=True)
assert (REPO / "ai" / "prepare_dataset.py").exists(), "Không thấy ai/prepare_dataset.py — kiểm tra repo_dir / repo_url"
sys.path.insert(0, str(REPO / "ai")); sys.path.insert(0, str(REPO / "pi"))
import ultralytics; print("ultralytics", ultralytics.__version__)

## 2. Lấy dataset từ Drive và chia train/val/test THEO HỘP (AI-02)

In [ ]:
import zipfile
RAW = WORK / "dataset"
src_zip = Path(CFG["drive_dir"]) / "dataset.zip"
src_dir = Path(CFG["drive_dir"]) / "dataset"
if src_zip.exists():
    shutil.rmtree(RAW, ignore_errors=True)
    with zipfile.ZipFile(src_zip) as zf:
        zf.extractall(WORK)
    # zip có thể bọc thêm một cấp thư mục
    if not RAW.exists():
        cand = [p for p in WORK.iterdir() if p.is_dir() and any((p / c).is_dir() for c in ("ok", "defect", "tear"))]
        assert cand, "Trong zip không thấy thư mục lớp ok/…"
        cand[0].rename(RAW)
elif src_dir.exists():
    RAW = src_dir
else:
    raise SystemExit(f"Không thấy {src_zip} hoặc {src_dir}")

from prepare_dataset import prepare, print_summary
DATA = WORK / "data"
summary = prepare(RAW, DATA, CFG["classes"], seed=CFG["seed"])
print_summary(summary)

## 3. Train YOLOv8n-cls (AI-03)
imgsz 224, 50 epoch, patience 10; augmentation: xoay ±15°, lật ngang 0.5, đổi độ sáng (hsv_v 0.4), scale 0.1.

In [ ]:
from train_yolo_cls import TrainCfg, train, export_onnx
tcfg = TrainCfg(data=str(DATA), out=str(WORK / "runs" / "yolo_cls"), model=CFG["model"], imgsz=CFG["imgsz"],
                epochs=CFG["epochs"], patience=CFG["patience"], batch=CFG["batch"], seed=CFG["seed"], version=CFG["version"])
model, run_dir = train(tcfg)
print("best.pt:", run_dir / "weights" / "best.pt")

In [ ]:
# Đường cong train và confusion matrix do ultralytics vẽ
from IPython.display import Image as IImage, display
for name in ("results.png", "confusion_matrix.png"):
    p = run_dir / name
    if p.exists():
        display(IImage(filename=str(p), width=700))

## 4. Export ONNX (imgsz 224, opset 12) + labels.json + model_card.json (AI-04)

In [ ]:
paths = export_onnx(model, run_dir, tcfg)
EXPORT = paths["onnx"].parent
print(json.dumps(json.load(open(paths["labels"], encoding="utf-8")), ensure_ascii=False))
print(open(paths["model_card"], encoding="utf-8").read()[:800])

## 5. Đánh giá trên test + chọn ngưỡng an toàn trên val (AI-05, AI-07)
Chạy qua đúng `OnnxClassifier` của Pi → tiền xử lý giống lúc triển khai. Ngưỡng được ghi vào `model_card.json`.

In [ ]:
from evaluate import evaluate
res = evaluate(paths["onnx"], DATA, EXPORT, None, None, CFG["target_recall"], CFG["min_confidence"], update_card=True, threads=2)
for name in ("confusion_matrix.png", "misclassified_grid.png"):
    p = EXPORT / name
    if p.exists():
        display(IImage(filename=str(p), width=600))

## 6. Benchmark nhanh bằng onnxruntime (số thật phải đo trên Pi bằng `pi/tools/benchmark_model.py`)

In [ ]:
import time, numpy as np, onnxruntime as ort
so = ort.SessionOptions(); so.intra_op_num_threads = 4
sess = ort.InferenceSession(str(paths["onnx"]), so, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
x = np.random.rand(1, 3, CFG["imgsz"], CFG["imgsz"]).astype(np.float32)
for _ in range(10): sess.run(None, {inp.name: x})
ts = []
for _ in range(100):
    t0 = time.perf_counter(); sess.run(None, {inp.name: x}); ts.append((time.perf_counter() - t0) * 1000)
print(f"Colab CPU: mean {np.mean(ts):.2f} ms | p95 {np.percentile(ts, 95):.2f} ms | model {paths['onnx'].stat().st_size/2**20:.2f} MB")

## 7. Chép kết quả về Drive — 3 file cần cho Pi: `packaging_cls.onnx`, `labels.json`, `model_card.json`

In [ ]:
import datetime
dst = Path(CFG["drive_dir"]) / f"export_yolo_{datetime.datetime.now():%Y%m%d_%H%M}"
dst.mkdir(parents=True, exist_ok=True)
for f in list(EXPORT.iterdir()) + [run_dir / "weights" / "best.pt"]:
    if f.is_file():
        shutil.copy2(f, dst / f.name)
print("Đã lưu:", dst, sorted(p.name for p in dst.iterdir()))
print("Trên Pi: scp 3 file vào /opt/sorter/models/ rồi đặt ai.backend: onnx trong /etc/sorter/config.yaml (xem ai/README.md)")